# DIHE Training (Fast I/O Mode)

This notebook copies datasets to Colab's local SSD for faster training, patches legacy `torchvision` issues, and runs the training pipeline.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Dependencies
!pip install ray[tune] click scikit-image networkx squarify

In [ ]:
# 3. GLOBAL COMPATIBILITY PATCH (Torchvision 0.9 fix)
import sys
import types
import torch
import torchvision.models

fake_utils = types.ModuleType("torchvision.models.utils")
if hasattr(torch.hub, 'load_state_dict_from_url'):
    fake_utils.load_state_dict_from_url = torch.hub.load_state_dict_from_url
else:
    from torch.hub import load_state_dict_from_url
    fake_utils.load_state_dict_from_url = load_state_dict_from_url

sys.modules["torchvision.models.utils"] = fake_utils
torchvision.models.utils = fake_utils
print("✅ Patch applied.")

In [ ]:
# 4. COPY DATASETS (The Fast Way)
import os
import time

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/dihe'
LOCAL_DATA_ROOT = '/content/datasets'
ZIP_PATH_DRIVE = os.path.join(DRIVE_PROJECT_ROOT, 'datasets.zip')
ZIP_PATH_LOCAL = os.path.join(LOCAL_DATA_ROOT, 'datasets.zip')

# Add code folder to path
import sys
if DRIVE_PROJECT_ROOT not in sys.path:
    sys.path.append(DRIVE_PROJECT_ROOT)

print("⏳ Copying compressed dataset from Drive...")
start_time = time.time()

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

if os.path.exists(ZIP_PATH_DRIVE):
    # 1. Copy the single zip file
    !cp "{ZIP_PATH_DRIVE}" "{ZIP_PATH_LOCAL}"

    # 2. Unzip locally (-q is crucial to suppress printing 100k lines)
    print("⏳ Unzipping...")
    !unzip -q "{ZIP_PATH_LOCAL}" -d "{LOCAL_DATA_ROOT}"

    print(f"✅ Copy & Unzip complete in {time.time() - start_time:.2f} seconds.")
else:
    print(f"❌ Zip file not found at {ZIP_PATH_DRIVE}. Please zip your datasets folder on Drive first.")


In [ ]:
try:
    # The package is named 'cvpce', so we import directly from it
    from cvpce import datautils, classification_training, classification_eval
    from cvpce.models import classification
    print("✅ Modules loaded.")
except ImportError as e:
    print(f"❌ Error loading modules: {e}")

In [ ]:
# 5. Configure Paths (Using LOCAL paths for Data, DRIVE paths for Output)

# --- GROCERY PRODUCTS (Local) ---
GP_ROOT = os.path.join(LOCAL_DATA_ROOT, 'Grocery_products')
GP_TRAIN_FOLDERS = [os.path.join(GP_ROOT, 'Training')]
GP_TEST_DIR = os.path.join(GP_ROOT, 'Testing')
GP_ANN_DIR = os.path.join(GP_ROOT, 'Annotations')

# --- SKU110K (Local) ---
SKU_IMG_DIR = os.path.join(LOCAL_DATA_ROOT, 'SKU110K', 'images')
SKU_ANN_FILE = os.path.join(LOCAL_DATA_ROOT, 'SKU110K', 'annotations', 'annotations_train.csv')

# --- OUTPUT (Drive - So results are saved permanently) ---
OUT_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

SKU110K_SKIP = ['test_274.jpg', 'train_882.jpg', 'train_924.jpg', 'train_4222.jpg', 'train_5822.jpg',
    'train_789.jpg', 'train_5007.jpg', 'train_6090.jpg', 'train_7576.jpg',
    'train_104.jpg', 'train_890.jpg', 'train_1296.jpg', 'train_3029.jpg', 'train_3530.jpg', 'train_3622.jpg', 'train_4899.jpg', 'train_6216.jpg', 'train_7880.jpg',
    'train_701.jpg', 'train_6566.jpg']

## Step 6: Pretrain GAN

In [ ]:
pretrain_opts = classification_training.ClassificationTrainingOptions()

pretrain_opts.dataset = datautils.GroceryProductsDataset(GP_TRAIN_FOLDERS, include_masks=False)
pretrain_opts.discriminatorset = datautils.TargetDomainDataset(SKU_IMG_DIR, SKU_ANN_FILE, skip=SKU110K_SKIP)

pretrain_opts.output_path = OUT_DIR
pretrain_opts.batch_size = 16
pretrain_opts.num_workers = 4 # Increased workers since local I/O is fast
pretrain_opts.epochs = 50
pretrain_opts.masks = False

print("Starting GAN Pretraining...")
classification_training.pretrain_gan(pretrain_opts)

In [ ]:
import torchvision.models.vgg as vgg

# Manually restore the model_urls dictionary expected by legacy code
if not hasattr(vgg, 'model_urls'):
    vgg.model_urls = {
        'vgg16': 'https://download.pytorch.org/models/vgg16-397923af.pth',
        'vgg16_bn': 'https://download.pytorch.org/models/vgg16_bn-6c64b313.pth',
        'vgg11': 'https://download.pytorch.org/models/vgg11-bbd30ac9.pth',
        'vgg13': 'https://download.pytorch.org/models/vgg13-c768596a.pth',
        'vgg19': 'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth',
        'vgg19_bn': 'https://download.pytorch.org/models/vgg19_bn-c79401a0.pth',
    }
    print("✅ Patched torchvision.models.vgg.model_urls")
else:
    print("ℹ️ torchvision.models.vgg.model_urls already exists.")

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler

# 1. Save the original class just in case
if not hasattr(lr_scheduler, 'OriginalMultiplicativeLR'):
    lr_scheduler.OriginalMultiplicativeLR = lr_scheduler.MultiplicativeLR

# 2. Define a wrapper that accepts 'verbose' but ignores it
class PatchedMultiplicativeLR(lr_scheduler.OriginalMultiplicativeLR):
    def __init__(self, optimizer, lr_lambda, last_epoch=-1, verbose=False):
        # Call the original __init__ WITHOUT the verbose argument
        super().__init__(optimizer, lr_lambda, last_epoch=last_epoch)

# 3. Apply the patch to the library
lr_scheduler.MultiplicativeLR = PatchedMultiplicativeLR

print("✅ Patched MultiplicativeLR to ignore the removed 'verbose' argument.")

In [ ]:
import os

TEST_ROOT = '/content/datasets/Grocery_products/Testing'

print(f"📂 Scanning {TEST_ROOT} to fix filenames...")

# Loop through store1, store2, etc.
for store_folder in os.listdir(TEST_ROOT):
    store_path = os.path.join(TEST_ROOT, store_folder)

    # Ensure it's a folder (e.g., 'store2')
    if os.path.isdir(store_path) and 'store' in store_folder:
        images_dir = os.path.join(store_path, 'images')

        if os.path.exists(images_dir):
            count = 0
            for filename in os.listdir(images_dir):
                # Check if file is "18.jpg" (Bad) instead of "store2_18.jpg" (Good)
                # We check if the filename does NOT start with the folder name
                if filename.endswith('.jpg') and not filename.startswith(store_folder):

                    old_path = os.path.join(images_dir, filename)

                    # specific fix for the "s" vs "store" naming convention if needed
                    # But based on your error, it wants 'store2_18.jpg'
                    new_name = f"{store_folder}_{filename}"
                    new_path = os.path.join(images_dir, new_name)

                    os.rename(old_path, new_path)
                    count += 1

            if count > 0:
                print(f"✅ {store_folder}: Renamed {count} images (e.g., '18.jpg' -> '{store_folder}_18.jpg')")
            else:
                print(f"ℹ️ {store_folder}: Files already correct.")

print("\n🎉 Filenames fixed. You can resume training.")

## Step 7: Train DIHE

In [ ]:
train_opts = classification_training.ClassificationTrainingOptions()

train_opts.dataset = datautils.GroceryProductsDataset(GP_TRAIN_FOLDERS, include_annotations=True, include_masks=False)
train_opts.discriminatorset = datautils.TargetDomainDataset(SKU_IMG_DIR, SKU_ANN_FILE, skip=SKU110K_SKIP)
train_opts.evalset = datautils.GroceryProductsTestSet(GP_TEST_DIR, GP_ANN_DIR, only=None)
train_opts.evaldata = train_opts.dataset

train_opts.output_path = OUT_DIR
train_opts.batch_size = 4
train_opts.num_workers = 4
train_opts.epochs = 30
train_opts.gpus = 1 if torch.cuda.is_available() else 0
train_opts.batchnorm = False

train_opts.load_gan = os.path.join(OUT_DIR, 'checkpoint.tar')
train_opts.load_encoder = None

if os.path.exists(train_opts.load_gan):
    print("Starting DIHE Training...")
    classification_training.train_dihe(0, train_opts)
else:
    print(f"GAN Checkpoint not found at {train_opts.load_gan}")

## Step 8: Evaluation

In [ ]:
MODEL_TYPE = 'vgg16'
BATCH_NORM = False
ENC_WEIGHTS = os.path.join(OUT_DIR, 'epoch_9.tar')

if os.path.exists(ENC_WEIGHTS):
    if MODEL_TYPE == 'vgg16':
        encoder = classification.macvgg_embedder(model='vgg16_bn' if BATCH_NORM else 'vgg16', pretrained=True).cuda()
    elif MODEL_TYPE == 'resnet50':
        encoder = classification.macresnet_encoder(pretrained=False).cuda()

    state = torch.load(ENC_WEIGHTS)
    encoder.load_state_dict(state['model_state_dict'])
    encoder.eval()

    sampleset = datautils.GroceryProductsDataset(GP_TRAIN_FOLDERS, include_annotations=True)
    testset = datautils.GroceryProductsTestSet(GP_TEST_DIR, GP_ANN_DIR)

    print("Running Evaluation...")
    accuracy = classification_eval.eval_dihe(encoder, sampleset, testset, batch_size=8, num_workers=4, k=(1, 5, 10))
    print("Final Accuracy:", accuracy)
else:
    print(f"Checkpoint not found at {ENC_WEIGHTS}")